# LayoutLMv3 on FUNSD: proxy for DepositTrack extraction pass A

Run this on a **free GPU runtime**: Colab → Runtime → Change runtime type → T4 GPU. The free tier is enough; no paid plan is needed.

**Why FUNSD, and what this does and does not show.** There is no public dataset of labelled FDRs. FUNSD (Jaume et al., 2019) holds 199 real scanned forms whose words are labelled as *question* (a caption such as "FDR No"), *answer* (its value) and *header*: the same caption-and-value layout an FDR has. Fine-tuning LayoutLMv3 here measures the method on real documents. The model cannot read FDR fields (FDR number, IFSC, maturity date), so **the API does not load it**. Live extraction stays on OCR + configured rules (pass B) until labelled real FDRs exist, and then the same recipe can be pointed at them.

What it does:
1. Loads FUNSD from Hugging Face (`nielsr/funsd-layoutlmv3`: 149 train / 50 test forms; research and educational use, cite the paper).
2. Fine-tunes `microsoft/layoutlmv3-base` for token classification with the paper's FUNSD settings (learning rate 1e-5, batch 16, 1,000 steps; Huang et al., 2022). On a T4 the batch of 16 is reached as 4 forms per step with gradients accumulated over 4 steps, so it fits in memory.
3. Evaluates the model after the 1,000 steps once on the FUNSD test split (no checkpoint is picked on test) and reports entity-level F1 (**a proxy result on forms, not FDRs; label it as such**). Forms longer than 512 tokens are truncated; the words cut off are counted, and a second score treats them as missed.
4. Saves `layoutlmv3-funsd-proxy.zip` with `metrics.json`, for the record. Do **not** unzip it into `models/layoutlmv3-fdr/`.

**If a cell fails with `CUDA out of memory`:** choose Runtime → Restart session (the failed run keeps GPU memory allocated), lower `PER_DEVICE_BATCH_SIZE` in the settings cell (for example to 2; `GRADIENT_ACCUMULATION_STEPS` then adjusts so the effective batch stays 16), and run all cells again.

In [ ]:
# Settings - change only if needed
MAX_STEPS = 1000  # optimizer steps, as in the paper
LEARNING_RATE = 1e-5  # LayoutLMv3 paper, FUNSD
EFFECTIVE_BATCH_SIZE = 16  # LayoutLMv3 paper, FUNSD
PER_DEVICE_BATCH_SIZE = 4  # fits a 15 GB T4; lower to 2 if CUDA runs out of memory
GRADIENT_ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // PER_DEVICE_BATCH_SIZE
if PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS != EFFECTIVE_BATCH_SIZE:
    raise ValueError(
        "PER_DEVICE_BATCH_SIZE must divide EFFECTIVE_BATCH_SIZE (use 1, 2, 4, 8 or 16)"
    )
SEED = 7  # set before the model is created, so the classification head is reproducible

In [ ]:
!nvidia-smi
# seqeval is not installed: it ships only as a source package that fails to build on current
# Colab images. Entity-level F1 is computed in the next cell with the same rules instead.
!pip -q install --prefer-binary transformers datasets accelerate

In [ ]:
# Entity-level precision / recall / F1 (same rules as seqeval's default, conlleval mode)
def entities(labels):
    """Entity spans (type, start, end) from one BIO label sequence, as seqeval's default mode
    (conlleval) reads them: an entity starts at B-X, or at I-X after anything that is not X."""
    spans, kind, start = [], None, None
    for i, label in enumerate([*labels, "O"]):
        prefix, _, tag = label.partition("-")
        starts = prefix == "B" or (prefix == "I" and tag != kind)
        if kind is not None and (prefix == "O" or starts):
            spans.append((kind, start, i - 1))
            kind = None
        if starts:
            kind, start = tag, i
    return spans


def entity_scores(true_seq, pred_seq):
    """Micro precision / recall / F1 over exact-match entities, plus a per-type breakdown."""
    gold, found = set(), set()
    for n, (t, p) in enumerate(zip(true_seq, pred_seq, strict=True)):
        gold |= {(n, *e) for e in entities(t)}
        found |= {(n, *e) for e in entities(p)}

    def prf(g, f):
        hit = len(g & f)
        precision = hit / len(f) if f else 0.0
        recall = hit / len(g) if g else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        return {"precision": precision, "recall": recall, "f1": f1, "support": len(g)}

    per_type = {
        kind: prf({e for e in gold if e[1] == kind}, {e for e in found if e[1] == kind})
        for kind in sorted({e[1] for e in gold | found})
    }
    return prf(gold, found), per_type

In [ ]:
from datasets import load_dataset

funsd = load_dataset("nielsr/funsd-layoutlmv3")
LABELS = funsd["train"].features["ner_tags"].feature.names
label2id = {name: i for i, name in enumerate(LABELS)}
id2label = dict(enumerate(LABELS))
print(funsd, LABELS)

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)


def encode(batch):
    images = [image.convert("RGB") for image in batch["image"]]
    return processor(
        images,
        batch["tokens"],
        boxes=batch["bboxes"],
        word_labels=batch["ner_tags"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )


columns = funsd["train"].column_names
train_ds = funsd["train"].map(encode, batched=True, remove_columns=columns)
test_ds = funsd["test"].map(encode, batched=True, remove_columns=columns)
train_ds.set_format("torch")
test_ds.set_format("torch")

In [ ]:
import torch
from transformers import (
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(SEED)  # before from_pretrained: the new classification head is randomly initialised
model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=len(LABELS), id2label=id2label, label2id=label2id
)


def sequences(predicted, gold):
    true_seq, pred_seq = [], []
    for p_row, g_row in zip(predicted, gold, strict=True):
        pairs = [
            (id2label[int(g)], id2label[int(p)])
            for p, g in zip(p_row, g_row, strict=True)
            if g != -100
        ]
        true_seq.append([g for g, _ in pairs])
        pred_seq.append([p for _, p in pairs])
    return true_seq, pred_seq


def metrics(pred):
    true_seq, pred_seq = sequences(pred.predictions.argmax(-1), pred.label_ids)
    overall, _ = entity_scores(true_seq, pred_seq)
    return {"entity_f1": overall["f1"]}


args = TrainingArguments(
    output_dir="out",
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    eval_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,  # move eval outputs to CPU in chunks
    # Paper protocol: a fixed 1,000 steps, then one evaluation on test. No evaluation or
    # checkpoint selection during training (FUNSD has no validation split), and no
    # checkpoints written (they would only fill the Colab disk).
    eval_strategy="no",
    save_strategy="no",
    fp16=torch.cuda.is_available(),  # fp16 needs CUDA; on CPU it would raise
    seed=SEED,
    report_to=[],
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=metrics
)
torch.cuda.empty_cache()
trainer.train()
result = trainer.evaluate()
print(result)

Note: FUNSD has no separate validation split, so nothing is selected on the test split: the model after the fixed 1,000 steps (the paper's FUNSD protocol) is evaluated once. Forms longer than 512 tokens are truncated, so the next cell also reports the score with the cut-off words counted as predicted `O` (their entities then count as missed).

In [ ]:
# Per-label report on the FUNSD test split (proxy result)
pred = trainer.predict(test_ds)
true_seq, pred_seq = sequences(pred.predictions.argmax(-1), pred.label_ids)
overall, per_type = entity_scores(true_seq, pred_seq)

# Words past the 512-token limit carry no label in test_ds. Score again on every word of each
# form, with the cut-off words predicted "O", so their entities count as missed. The tokenizer
# call repeats the one in encode(), so its word_ids give the word behind each labelled token.
test_words = funsd["test"].select_columns(["tokens", "bboxes", "ner_tags"])
gold_words, pred_all, cut_off = [], [], []
for row, kept_pred in zip(test_words, pred_seq, strict=True):
    ids = processor.tokenizer(
        row["tokens"], boxes=row["bboxes"], truncation=True, max_length=512
    ).word_ids()
    labelled = list(dict.fromkeys(w for w in ids if w is not None))  # first token per word
    words = ["O"] * len(row["tokens"])
    for w, label in zip(labelled, kept_pred, strict=True):
        words[w] = label
    gold_words.append([id2label[t] for t in row["ner_tags"]])
    pred_all.append(words)
    cut_off.append(len(row["tokens"]) - len(labelled))
overall_all, per_type_all = entity_scores(gold_words, pred_all)
truncation = {
    "test_words": sum(len(g) for g in gold_words),
    "test_words_not_scored": sum(cut_off),
    "test_forms_with_words_not_scored": sum(n > 0 for n in cut_off),
}
print(truncation)
print(f"{'entity':<10} {'precision':>9} {'recall':>7} {'f1':>7} {'support':>8}")
for kind, s in [*per_type.items(), ("overall", overall)]:
    print(
        f"{kind:<10} {s['precision']:>9.4f} {s['recall']:>7.4f} {s['f1']:>7.4f} {s['support']:>8}"
    )
print(f"overall F1 with cut-off words counted as missed: {overall_all['f1']:.4f}")

In [ ]:
import json
import shutil

trainer.save_model("layoutlmv3-funsd-proxy")
processor.save_pretrained("layoutlmv3-funsd-proxy")
with open("layoutlmv3-funsd-proxy/metrics.json", "w") as fh:
    json.dump(
        {
            "data": "FUNSD (Jaume et al., 2019), scanned forms - proxy, not FDRs",
            "entity_f1": result.get("eval_entity_f1"),
            "test_entity_scores": overall,
            "test_entity_scores_by_type": per_type,
            "test_entity_scores_all_words": overall_all,
            "test_entity_scores_all_words_by_type": per_type_all,
            "truncation": {"max_length": 512, **truncation},
            "train_forms": len(funsd["train"]),
            "test_forms": len(funsd["test"]),
            "max_steps": MAX_STEPS,
            "learning_rate": LEARNING_RATE,
            "effective_batch_size": EFFECTIVE_BATCH_SIZE,
            "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "seed": SEED,
            "fp16": bool(args.fp16),
            "checkpoint_selected_on": "none: the model after max_steps, evaluated once on test",
            "live_extraction": False,
        },
        fh,
        indent=2,
    )
shutil.make_archive("layoutlmv3-funsd-proxy", "zip", "layoutlmv3-funsd-proxy")
try:
    from google.colab import files

    files.download("layoutlmv3-funsd-proxy.zip")
except ImportError:
    print("Download layoutlmv3-funsd-proxy.zip from the file browser.")